# Finite-Difference Playground

Validate one finite-difference kernel by comparing stencil weights, symbolic code, and numerical samples.

Navigation: [Index](../index.ipynb) | Previous: [Boundary Conditions and Convergence](boundary_conditions_and_convergence.ipynb) | Next: [Wave Equation and C Code Generation](../3-wave_equation/wave_equation_and_c_codegen.ipynb)


## Validate One Kernel
The polynomial `x**4` has exact second derivative `12*x**2`. The fourth-order stencil should reproduce it at the sample point used here.

## Import SymPy for the Polynomial Test

These imports expose the NRPy and Python tools used in the next steps.


In [ ]:
import sympy as sp


## Import Stencil and Codegen Helpers

These imports expose the NRPy registries and infrastructure writers used below.


In [ ]:
import nrpy.c_codegen as ccg
import nrpy.finite_difference as fd
import nrpy.grid as grid
import nrpy.indexedexp as ixp
import nrpy.params as par


## Clear Playground Grid State

The reset clears tutorial-owned finite-difference helpers and fixes the order used in the next calculation.


In [ ]:
grid.glb_gridfcs_dict.pop("uu", None)
for name in list(fd.FDFunctions_dict):
    if name.startswith("fdD"):
        fd.FDFunctions_dict.pop(name, None)
par.set_parval_from_str("Infrastructure", "BHaH")
par.set_parval_from_str("finite_difference::fd_order", 4)


## Compute the Chosen Stencil Weights

The printed coefficients define the finite-difference stencil.


In [ ]:
coeffs, stencil = fd.compute_fdcoeffs_fdstencl("dDD00", 4)
print("stencil weights:")
for coeff, point in zip(coeffs, stencil):
    print(coeff, point)


## Test the Stencil on a Polynomial

A zero residual confirms that the symbolic expression matches the expected identity.


In [ ]:
x0 = sp.Rational(3, 2)
dx = sp.Rational(1, 5)
stencil_value = sp.simplify(
    sum(coeffs[i] * (x0 + stencil[i][0] * dx) ** 4 / dx**2 for i in range(len(coeffs)))
)
exact_value = sp.diff(sp.Symbol("x") ** 4, sp.Symbol("x"), 2).subs(sp.Symbol("x"), x0)
print("stencil value:", stencil_value)
print("exact value:", exact_value)
print("residual:", sp.simplify(stencil_value - exact_value))
if sp.simplify(stencil_value - exact_value) != 0:
    raise RuntimeError("Expected the stencil residual to vanish.")


## Emit a Generated Laplacian Assignment

The registry records named grid fields and their roles in generated code.


In [ ]:
uu = grid.register_gridfunctions("uu", group="EVOL")[0]
uu_dDD = ixp.declarerank2("uu_dDD", symmetry="sym01")
print("complete generated kernel:")
print(
    ccg.c_codegen(
        uu_dDD[0][0],
        "laplacian_x0",
        include_braces=False,
        verbose=False,
        enable_fd_codegen=True,
        enable_fd_functions=True,
    )
)
print("complete helper code:")
print(fd.construct_FD_functions_prefunc())


The residual check ties the displayed stencil to an exact polynomial derivative. The generated kernel and helper code show the same coefficients in the form used inside C loops.


## Continue to Wave-Equation Codegen
- [C Code Generation](../1-intro/c_codegen.ipynb)
- [Finite Differences](../1-intro/finite_difference.ipynb)
- [Wave Equation and C Code Generation](../3-wave_equation/wave_equation_and_c_codegen.ipynb)
